<a href="https://colab.research.google.com/github/carlospucv/recommender_system/blob/rs_v4/RS_UsandoImplicit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Modelo basado en filtrado colaborativo con factorización de matrices (Embeddings)
Usando una librería especializada como **Implicit** o bien utilizando el módulo de **Surprise**.

Estas librerías están diseñadas precisamente para problemas de recomendación que generalmente ofrecen resultados mucho mejores, con menos esfuerzo. En comparación rs_v3

Ventajas:



*   Más simple y robusto.
*   Resultados mucho más efectivos para datos pequeños o medianos.
*   No lidias directamente con alta dimensionalidad ni extrema esparsidad.








# ⚙️ Paso 1: Instalación
En Google Colab:

In [ ]:
!pip install implicit


#📁 Paso 2: Preparar datos (Código funcional)
Cargar los CSV's:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Librerías
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
import implicit

# Cargar datos
movies_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movies.csv')
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/resultados.csv')

# Limpieza inicial (eliminar listas muy cortas)
data['ID_Lista'] = data['ID_Lista'].astype(str)
data['ID_Pelicula'] = data['ID_Pelicula'].astype(str)

# Filtrar listas pequeñas (al menos 2 películas por lista)
valid_lists = data.groupby('ID_Lista').filter(lambda x: len(x) >= 2)

# Crear IDs categóricos y mappings correctos
user_categorical = valid_lists['ID_Lista'].astype('category')
movie_categorical = valid_lists['ID_Pelicula'].astype('category')

user_ids = user_categorical.cat.codes
movie_ids = movie_categorical.cat.codes

# Diccionarios claros (no confundir estos)
user_id_to_idx = dict(zip(user_categorical.cat.categories, range(len(user_categorical.cat.categories))))
movie_id_to_idx = dict(zip(movie_categorical.cat.categories, range(len(movie_categorical.cat.categories))))

idx_to_movie_id = {v: k for k, v in movie_id_to_idx.items()}

# Crear matriz Usuario-Item (interacciones) CSR
interaction_matrix = coo_matrix(
    (np.ones(len(valid_lists)), (user_ids, movie_ids)),
    shape=(len(user_id_to_idx), len(movie_id_to_idx))
).tocsr()


# 📦 Paso 3: Entrenar modelo ALS (Alternating Least Squares)
Este modelo es extremadamente eficiente y robusto:

In [ ]:
# Crear matriz Item-Usuario (para entrenamiento) CSR
item_user_matrix = interaction_matrix.T.tocsr()

# Modelo ALS (entrenar con item-user)
model = implicit.als.AlternatingLeastSquares(
    factors=50, iterations=30, regularization=0.01, random_state=42
)
model.fit(item_user_matrix)


# 🎯 Paso 4: Crear función de recomendaciones

In [ ]:
# Función recomendadora
def recomendar_peliculas(id_lista, N=5):
    if id_lista not in user_id_to_idx:
        raise ValueError(f"El ID {id_lista} no existe en los datos actuales.")

    user_idx = user_id_to_idx[id_lista]
    # Pasar SOLO LA FILA del usuario actual (1 fila exactamente)
    user_items = interaction_matrix[user_idx]

    # Obtener recomendaciones correctamente
    recommendations = model.recommend(
        userid=user_idx,
        user_items=user_items,
        N=N,
        filter_already_liked_items=True
    )

    movie_indices = [movie_idx for movie_idx, _ in recommendations]
    movie_real_ids = [idx_to_movie_id[idx] for idx in movie_indices]

    return movies_df[movies_df['id'].astype(str).isin(movie_real_ids)][['id', 'title']]

In [ ]:
# Revisa IDs de listas disponibles después del filtrado
ids_listas_disponibles = valid_lists['ID_Lista'].astype(str).unique()
print(ids_listas_disponibles[:20])  # muestra primeros 20 IDs válidos

# 🚀 Paso 5: Probar recomendaciones inmediatamente

In [ ]:
# Probar recomendaciones con un usuario válido
id_lista_prueba = valid_lists['ID_Lista'].iloc[0]  # ID garantizado real
print(f"Probando con ID_Lista: {id_lista_prueba}")

recomendaciones = recomendar_peliculas(id_lista_prueba)
print(recomendaciones)

# 📊 Paso 6: Evaluar el modelo
Evalúa de forma rápida, eficaz y directa la precisión del modelo usando listas reales:

In [ ]:
# Evaluación sencilla:
from implicit.evaluation import precision_at_k

precision = precision_at_k(model, interaction_matrix.T, interaction_matrix.T, K=5)
print(f'Precisión promedio @5: {precision.mean():.2%}')
